In [22]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [23]:
df = pd.read_csv('churn_data.csv')

In [40]:
print(df.head(5))
print(df.shape)
print(df.isnull().sum())
print(df.describe())

   customer_id  tenure_months  monthly_charges  total_charges  num_products  \
0            1             52       131.240541    3271.805095             2   
1            2             15       127.928582    6420.560653             1   
2            3             61        71.633859    1711.881514             4   
3            4             21       106.851068    4485.171099             4   
4            5             24        46.647958    5891.263239             2   

   num_support_calls   contract_type    payment_method  age  is_senior  \
0                  7        One year     Bank transfer   55          0   
1                  1        Two year       Credit card   57          0   
2                  0  Month-to-month       Credit card   18          0   
3                  6  Month-to-month  Electronic check   46          0   
4                  0        Two year     Bank transfer   27          1   

   churned  contract_type_encoded  payment_method_encoded  
0        0          

In [32]:
df['contract_type_encoded'] = LabelEncoder().fit_transform(df['contract_type'])
df['payment_method_encoded'] = LabelEncoder().fit_transform(df['payment_method'])

In [33]:
features = ['tenure_months','monthly_charges','total_charges','num_products','num_support_calls','contract_type_encoded','payment_method_encoded','age','is_senior']
X = df[features]
y = df['churned']

In [34]:
    # Preprocess the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
# Make predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]
# Evaluate the model
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC AUC Score: {roc_auc:.4f}')

Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
ROC AUC Score: 0.7517


In [35]:
# 检查预测结果分布
print("y_pred 分布:")
print(pd.Series(y_pred).value_counts())
print("\ny_test 分布:")
print(pd.Series(y_test).value_counts())
print("\ny_proba 统计:")
print(f"最小值: {y_proba.min():.4f}")
print(f"最大值: {y_proba.max():.4f}")
print(f"均值: {y_proba.mean():.4f}")
print(f"中位数: {np.median(y_proba):.4f}")

y_pred 分布:
0    197
1      3
Name: count, dtype: int64

y_test 分布:
churned
0    182
1     18
Name: count, dtype: int64

y_proba 统计:
最小值: 0.0000
最大值: 0.5700
均值: 0.1062
中位数: 0.0500


In [36]:
# 方案 1: 调整分类阈值
threshold = 0.3  # 降低阈值
y_pred_adjusted = (y_proba >= threshold).astype(int)

precision_adj = precision_score(y_test, y_pred_adjusted)
recall_adj = recall_score(y_test, y_pred_adjusted)
f1_adj = f1_score(y_test, y_pred_adjusted)

print(f"调整阈值到 {threshold} 后:")
print(f'Precision: {precision_adj:.4f}')
print(f'Recall: {recall_adj:.4f}')
print(f'F1 Score: {f1_adj:.4f}')
print(f'ROC AUC Score: {roc_auc:.4f}')

print(f"\n预测分布: {pd.Series(y_pred_adjusted).value_counts().to_dict()}")

调整阈值到 0.3 后:
Precision: 0.2963
Recall: 0.4444
F1 Score: 0.3556
ROC AUC Score: 0.7517

预测分布: {0: 173, 1: 27}


In [ ]:
# 测试 class_weight='balanced' 的效果
model_test = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,    
    random_state=42,
    class_weight='balanced'
)
model_test.fit(X_train_scaled, y_train)

y_pred_test = model_test.predict(X_test_scaled)
y_proba_test = model_test.predict_proba(X_test_scaled)[:, 1]

print("使用工作代码的参数:")
print(f'Precision: {precision_score(y_test, y_pred_test):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_test):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_test):.4f}')
print(f'ROC AUC Score: {roc_auc_score(y_test, y_proba_test):.4f}')

print(f"\n预测分布: {pd.Series(y_pred_test).value_counts().to_dict()}")
print(f"实际分布: {pd.Series(y_test).value_counts().to_dict()}")

使用工作代码的参数:
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
ROC AUC Score: 0.7952

预测分布: {0: 200}
实际分布: {0: 182, 1: 18}


c:\Users\huang\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
